In [1]:
# ==== Building the training dataset for multi-label classification (identifying grammar patterns) ==== #

In [2]:
# Plan for collecting data for wide variety of sentences for multi-label classification

# ~150 grammar per JLPT level (~150 labels total)
# Assuming 1-10 grammar points maximum, then how many sentences will I need total?

# Labels per sentence distribution (example):
# - 1 label:   20-30% of sentences
# - 2 labels:  35-45% of sentences (most common)
# - 3 labels:  20-25% of sentences
# - 4 labels:  5-10% of sentences
# - 5+ labels: 1-5% of sentences

# Label frequency distribution:
# - Common grammar (は, を, に, etc.): 200-500 examples
# - Medium grammar (ている, ます, etc.): 50-150 examples
# - Rare grammar (specialized patterns): 10-50 examples

Success
## Video Collection Plan

**Target:** 30 videos across 10 categories (3 per category)

| # | Category | Approx. Sentences | Why Important | Example Channels |
|---|----------|:-----------------:|---------------|------------------|
| 1 | Educational/Lectures | ~12K | Formal grammar, complete sentences, clear speech | TED日本語, NHK高校講座 |
| 2 | Daily Vlogs | ~10K | Casual grammar, conversational patterns | はじめしゃちょー, Fischer's |
| 3 | News/Documentary | ~15K | Formal grammar, complex constructions | NHKニュース, テレ東BIZ |
| 4 | Anime/Entertainment | ~12K | Mix of casual/formal, varied contexts | Official anime channels |
| 5 | Cooking Shows | ~8K | Instructional grammar, て-form heavy | きまぐれクック, Cooking with Dog |
| 6 | Gaming/Let's Plays | ~10K | Very casual, lots of particles and reactions | 兄者弟者, ポッキー |
| 7 | Interviews/Talk Shows | ~12K | Natural conversation, question forms | ゴッドタン, しくじり先生 |
| 8 | How-To/Tutorials | ~10K | Imperative forms, explanatory grammar | メンタリストDaiGo, QuizKnock |
| 9 | Travel/Culture | ~9K | Descriptive grammar, storytelling | Abroad in Japan, TabiEats |
| 10 | Business/Finance | ~12K | Formal, conditional forms | 中田敦彦のYouTube大学 |
| | **Total** | **~110K** | | |

**After stratified sampling:** 5,000–10,000 high-quality annotated sentences

In [3]:
# ===== IMPORTS ===== #
import re
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from jlpt_grammar_parser import parse_jlpt_grammar_pdf, print_stats, extract_grammar_patterns, extract_all_grammar_patterns


In [4]:
# # ===== 10 Example Sentences per Grammar Point from JLPTSensei PDFs ===== #
# # HOWEVER, MAY NOT BE USING THESE SENTENCES
# #from jlpt_grammar_parser import parse_jlpt_grammar_pdf, print_stats # IMPORTED ABOVE
# #import pandas as pd

# # Define your PDF paths
# pdfs = {
#     'N5': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N5 Grammar Master Ebok by JLPTsensei.com.pdf',
#     'N4': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N4 Grammar Master Ebook by JLPTsensei.com.pdf',
#     'N3': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N3 Grammar Master Ebook by JLPTsensei.com.pdf',
#     'N2': '/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPT N2 Grammar Master Ebook by JLPTsensei.com.pdf',
# }

# # all_sentences = []
# # for level, path in pdfs.items():
#     # result = parse_jlpt_grammar_pdf(path, level=level)
#     # print_stats(result)
#     # all_sentences.extend(result['sentences'])

# # df = pd.DataFrame(all_sentences)
# # print(f"Total: {len(df)} sentences")
# patterns = extract_all_grammar_patterns(pdfs)

# # Convert to DataFrame
# df = pd.DataFrame(patterns)

In [5]:
def is_good_sentence(sentence):
    """Filter sentences suitable for N4 grammar training"""
    # Length: 5-50 characters (not too short, not too long)
    if not (10 <= len(sentence) <= 100):
        return False
    
    # Has ending punctuation
    if not re.search(r'[。！？!?]$', sentence):
        return False
    
    # Not just a single word
    if len(sentence) < 5:
        return False
    
    # Contains hiragana/katakana (not just kanji)
    if not re.search(r'[ぁ-ん]', sentence):
        return False
    
    return True

# This might reduce 5,000 → 3,500-4,000 high-quality sentences

In [20]:
# df
# df.to_csv("JLPTSensei_Grammars_All_JLPTs.csv")

In [21]:
# Store in CSV so I don't have to rerun the parser a lot
#df.to_csv("N5-N2_JLPTSensei.csv")
df = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N5-N2_JLPTSensei.csv")

In [ ]:
# Display all dataframes for each JLPT level and export them to JSON with the format:
# N5_grammars = df[df["jlpt_level"] == "N5"]
# N4_grammars = df[df["jlpt_level"] == "N4"]
# N3_grammars = df[df["jlpt_level"] == "N3"]
# N2_grammars = df[df["jlpt_level"] == "N2"]

# N5_grammars = N5_grammars.drop_duplicates(subset=["grammar_point"])
# N5_grammars = N5_grammars[['grammar_id', 'grammar_point', 'meaning']]

# N4_grammars = N4_grammars.drop_duplicates(subset=["grammar_point"])
# N4_grammars = N4_grammars[['grammar_id', 'grammar_point', 'meaning']]

# N3_grammars = N3_grammars.drop_duplicates(subset=["grammar_point"])
# N3_grammars = N3_grammars[['grammar_id', 'grammar_point', 'meaning']]

# N2_grammars = N2_grammars.drop_duplicates(subset=["grammar_point"])
# N2_grammars = N2_grammars[['grammar_id', 'grammar_point', 'meaning']]

# # Export them to CSVs
# N5_grammars.to_csv("N5_grammars_v1.csv")
# N4_grammars.to_csv("N4_grammars_v1.csv")
# N3_grammars.to_csv("N3_grammars_v1.csv")
# N2_grammars.to_csv("N2_grammars_v1.csv")


In [6]:
# split the dataframe into regex detectables and non-regex detectables.
# The non-regex detectables need a new index since the vector size is smaller

# Load all the dfs
N5_grammars = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N5_grammars_v1.csv")
N4_grammars = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N4_grammars_v1.csv")
N3_grammars = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N3_grammars_v1.csv")
N2_grammars = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N2_grammars_v1.csv")

N5_grammars_regex = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N5_regex_filtered.csv")
N4_grammars_regex = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N4_regex_filtered.csv")
N3_grammars_regex = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N3_regex_filtered.csv")
N2_grammars_regex = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/N2_regex_filtered.csv")

In [ ]:
# Filter all grammars that are no regex to get lists for all regex grammars for each level
N5_model_grammars = N5_grammars[~N5_grammars['grammar_point'].isin(N5_grammars_regex['grammar_point'])]
N4_model_grammars = N4_grammars[~N4_grammars['grammar_point'].isin(N4_grammars_regex['grammar_point'])]
N3_model_grammars = N3_grammars[~N3_grammars['grammar_point'].isin(N3_grammars_regex['grammar_point'])]
N2_model_grammars = N2_grammars[~N2_grammars['grammar_point'].isin(N2_grammars_regex['grammar_point'])]

# print(f"N5: {len(N5_grammars)} total → {len(N5_regex_grammars)} regex-detectable")
# print(f"N4: {len(N4_grammars)} total → {len(N4_regex_grammars)} regex-detectable")
# print(f"N3: {len(N3_grammars)} total → {len(N3_regex_grammars)} regex-detectable")
# print(f"N2: {len(N2_grammars)} total → {len(N2_regex_grammars)} regex-detectable")

# N5_model_grammars.to_csv("N5_grammars_model.csv", index=False)
# N4_model_grammars.to_csv("N4_grammars_model.csv", index=False)
# N3_model_grammars.to_csv("N3_grammars_model.csv", index=False)
# N2_model_grammars.to_csv("N2_grammars_model.csv", index=False)


In [11]:
N5_model_grammars = pd.read_csv("N5_grammars_model.csv", index_col=0)
N4_model_grammars = pd.read_csv("N4_grammars_model.csv", index_col=0)
N3_model_grammars = pd.read_csv("N3_grammars_model.csv", index_col=0)
N2_model_grammars = pd.read_csv("N2_grammars_model.csv", index_col=0)
N2_model_grammars.head(1)

,Unnamed: 0,grammar_id,grammar_point,meaning
2,3039,3,ばかり,"about, approximately~ Noun (indicates time or ..."


In [ ]:
# For all grammars for model. Create a new row to assign a new index.
# But keep the current index because I will use that as a database key.
# Adds a new 0-based sequential column without touching grammar_id
N5_model_grammars = N5_model_grammars.reset_index(drop=True)
#N5_model_grammars.insert(0, "model_idx", range(len(N5_model_grammars)))

N4_model_grammars = N4_model_grammars.reset_index(drop=True)
#N4_model_grammars.insert(0, "model_idx", range(len(N4_model_grammars)))

N3_model_grammars = N3_model_grammars.reset_index(drop=True)
#N3_model_grammars.insert(0, "model_idx", range(len(N3_model_grammars)))

N2_model_grammars = N2_model_grammars.reset_index(drop=True)
#N2_model_grammars.insert(0, "model_idx", range(len(N2_model_grammars)))
N5_model_grammars.head(1)

,model_idx,Unnamed: 0,grammar_id,grammar_point,meaning
0,0,10,2,だ・です,"to be (am, is, are, were, used to) present だ (..."


In [24]:
test_N5 = N5_model_grammars[["grammar_idx", "grammar_point", "meaning"]]
test_N5.to_csv("N5_model_grammars_reindexed.csv")

test_N4 = N4_model_grammars[["grammar_idx", "grammar_point", "meaning"]]
test_N4.to_csv("N4_model_grammars_reindexed.csv")

test_N3 = N3_model_grammars[["grammar_idx", "grammar_point", "meaning"]]
test_N3.to_csv("N3_model_grammars_reindexed.csv")

test_N2 = N2_model_grammars[["grammar_idx", "grammar_point", "meaning"]]
test_N2.to_csv("N2_model_grammars_reindexed.csv")

In [17]:
levels = {
    "N5": N5_model_grammars,
    "N4": N4_model_grammars,
    "N3": N3_model_grammars,
    "N2": N2_model_grammars,
}

for level, df in levels.items():
    unique = df["grammar_point"].unique()
    df["grammar_idx"] = df["grammar_point"].map({g: i for i, g in enumerate(unique)})
    print(f"{level}: {len(unique)} unique grammar points")


N5: 30 unique grammar points
N4: 34 unique grammar points
N3: 33 unique grammar points
N2: 24 unique grammar points


In [5]:
# Recommended distribution
categories = {
    'educational': 3,    # ~12K sentences
    'vlogs': 3,         # ~10K sentences
    'news': 3,          # ~15K sentences
    'anime': 3,         # ~12K sentences
    'cooking': 3,       # ~8K sentences
    'gaming': 3,        # ~10K sentences
    'interviews': 3,    # ~12K sentences
    'tutorials': 3,     # ~10K sentences
    'travel': 3,        # ~9K sentences
    'business': 3,      # ~12K sentences
}
# Total: ~110K sentences

In [30]:
# ===== READ, SAMPLE, FILTER, AND COMBINE PARSED SENTENCE CSVs ===== #

# ── Why 10% with temporal stratification? ───────────────────────────────────
# Simple random sampling can cluster sentences from the same part of a video.
# Instead, each video is split into N_BINS equal-position windows and
# SAMPLE_RATE is drawn from EACH bin — guaranteeing sentences from the start,
# middle, and end (maximum temporal context spread).
# 10% is the sweet spot: going higher (20-30%) adds annotation burden with
# diminishing diversity gains, and is_good_sentence will filter a portion out
# anyway, so effective yield is closer to 6-8% of raw sentences.

parsed_dir  = Path("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/parsed_sentences")
SAMPLE_RATE = 0.10  # 10% per bin
N_BINS      = 10    # Split each video into 10 equal temporal windows

all_dfs  = []
summary  = []

# Exclude the combined file — only process individual video CSVs
csv_files = sorted(f for f in parsed_dir.glob("*.csv") if f.name != "all_media_combined.csv")
print(f"Found {len(csv_files)} video CSV files\n")

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file, encoding="utf-8")
    except pd.errors.EmptyDataError:
        print(f"[SKIP] {csv_file.name}  (no columns to parse)")
        continue
    
    if df.empty or "text" not in df.columns:
        print(f"[SKIP] {csv_file.name}")
        continue

    total = len(df)

    # Build bin array externally — avoids adding a column and any index issues
    bins = (np.arange(total) / total * N_BINS).astype(int).clip(0, N_BINS - 1)

    # Stratified sample by temporal bin (group on the external array, not a column)
    sample = (
        df.groupby(bins, group_keys=False)
          .apply(lambda g: g.sample(frac=SAMPLE_RATE, random_state=42))
    )

    # Filter by quality
    sample = sample[sample["text"].apply(is_good_sentence)].copy()

    # Tag source video
    sample["source"] = csv_file.stem

    all_dfs.append(sample)
    summary.append({
        "file":            csv_file.name,
        "total":           total,
        "after_sampling":  int(total * SAMPLE_RATE),
        "after_filtering": len(sample),
    })

    print(f"[OK] {csv_file.name[:55]:55s}  {total:5d} → {int(total*SAMPLE_RATE):4d} sampled → {len(sample):4d} kept")

# ── Combine & clean up ───────────────────────────────────────────────────────
corpus_df = pd.concat(all_dfs, ignore_index=True)
corpus_df.drop(columns=["sentence_id", "start_time", "end_time", "jlpt_level", "grammar", "vocabulary", "source"], inplace=True, errors="ignore")
corpus_df.index.name = "sentence_id"

# ── Summary ──────────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary)
print(f"\n{'='*60}")
print(f"Videos loaded:             {len(summary_df)}")
print(f"Total sentences in corpus: {len(corpus_df)}")
print(f"Avg kept per video:        {len(corpus_df) // max(len(summary_df), 1)}")
print(f"{'='*60}")
print(summary_df.to_string(index=False))

corpus_df

Found 29 video CSV files

[OK] 1年間禁止生活について、謝罪させてください。.ja.csv                              196 →   19 sampled →   10 kept
[OK] Gachiakuta Episode 1 SUB⧸DUB ｜ The Sphere.ja.csv           178 →   17 sampled →   13 kept
[OK] Immersive Japanese Learning in Minecraft #：1 Effortless    393 →   39 sampled →   24 kept
[OK] New Saga Episode 1 SUB⧸DUB ｜ I'll Change My Fate.ja.csv    158 →   15 sampled →   13 kept
[OK] POP MARTで初めて買ってみたら、なんとも言えない結果になった🙂｜日本語ポッドキャスト.ja.csv       406 →   40 sampled →   21 kept
[OK] TERUのなるほどJapanese Podcast🎙️ #37 The Mindset of Taking t     64 →    6 sampled →    9 kept
[OK] 「日本に住む？メキシコに住む？」今年の振り返りを含めて！ (Japanese Radio for Listen    454 →   45 sampled →   33 kept
[SKIP] 【100万回再生】陳建一 シェフ 「究極の麻婆豆腐」｜赤坂四川飯店｜【中華】【鉄人】【プロの技】【最大のポイント】.ja.csv  (no columns to parse)
[OK] 【1hour Podcast】YUYUのダメなところ ⧸ ダメな自分とどう生きるか？ (Japanese Ra    794 →   79 sampled →   53 kept
[OK] 【Japanese Podcast】Hay Fever Season Is Here! - Master 70    530 →   53 sampled →   43 kept
[OK] 【LIVE】朝のニュース（Japan 

,text
sentence_id,
0,大きく言うとこの2点です。
1,あ、買ったものもダメなの?
2,これコンビニじゃない?
3,ごがコンビニ[笑い]ああ、量めっちゃ入ってる。
4,これもヒカルにやられたとか言ってますけども、僕の注意不足でございました。
...,...
576,もったいない使い方した。
577,あれ、一撃で遠くまで飛んでくるやつだな。
578,ああ、あそこでいるのね。


In [31]:
corpus_df.to_csv("corpus_v1.csv")

In [8]:
import math

In [9]:
# split the corpus into n batches to process separately because the Claude processor sucks
corpus_test = pd.read_csv("corpus_v1.csv")
n = 5

total      = len(corpus_test)
chunk_size = math.ceil(total / n)

print(f"Total rows:       {total}")
print(f"Rows per split:   {chunk_size}")

for i in range(n):
    part     = corpus_test.iloc[i * chunk_size : (i + 1) * chunk_size]
    out_name = f"corpus_v{i + 1}.csv"
    part.to_csv(out_name, index=False)
    print(f"Wrote {out_name}  ({len(part)} rows)")

Total rows:       581
Rows per split:   117
Wrote corpus_v1.csv  (117 rows)
Wrote corpus_v2.csv  (117 rows)
Wrote corpus_v3.csv  (117 rows)
Wrote corpus_v4.csv  (117 rows)
Wrote corpus_v5.csv  (113 rows)


In [11]:
# Recombine all the corpus.jsons into a dataframe called labeled_sents
import glob
import json

json_dir  = Path("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration")
json_files = sorted(json_dir.glob("corpus_v*.json"))

print(f"Found {len(json_files)} JSON files\n")

dfs = []
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        data = json.load(f)

    # Each key is a sentence, each value is a label array
    df = pd.DataFrame([
        {"text": sentence, "labels": label_array}
        for sentence, label_array in data.items()
    ])
    dfs.append(df)
    print(f"[OK] {json_file.name}  ({len(df)} rows)")

corpus_labeled = pd.concat(dfs, ignore_index=True)
corpus_labeled.index.name = "sentence_id"

print(f"\nTotal rows: {len(corpus_labeled)}")
corpus_labeled.head()

Found 5 JSON files

[OK] corpus_v1.json  (117 rows)
[OK] corpus_v2.json  (117 rows)
[OK] corpus_v3.json  (117 rows)
[OK] corpus_v4.json  (117 rows)
[OK] corpus_v5.json  (113 rows)

Total rows: 581


,text,labels
sentence_id,,
0,それはどっちかというと、あの家族家の話と言いますか。,"[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."
1,本当ご迷惑をしました。,"[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."
2,だっていきなりその出てたやつはどこで買ってきたかも言ってないし俺ら。,"[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."
3,はい、という感じでした。,"[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."
4,そんな形で僕のコンビニ禁止生活は心配に終わりましたが、これ幸いにもコンビニ禁止失敗したのが2...,"[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -..."


In [ ]:
corpus_labeled

In [15]:
# Drop all -1 vectors and reset index
corpus_labeled = corpus_labeled[corpus_labeled["labels"].apply(lambda x: -1 not in x)]
corpus_labeled = corpus_labeled.reset_index(drop=True)
corpus_labeled.index.name = "sentence_id"
corpus_labeled.to_csv("training_sent_labeled.csv")

In [6]:
jlpt_grammar_df = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/JLPTSensei_Grammars_All_JLPTs.csv")

In [13]:
#jlpt_grammar_df = jlpt_grammar_df.drop(columns=["furigana", "Unnamed: 0", "page"])
jlpt_N4_df = jlpt_grammar_df[jlpt_grammar_df["jlpt_level"] == "N4"].reset_index()
jlpt_N4_df = jlpt_N4_df.drop(columns=["index"])
jlpt_N4_df.to_csv("jlpt_n4_grammars_v1.csv")